## Phase 1: Build Human Design Knowledge Base (Week1)

* pymupdf extract pdfs
* cut chunks
* embedding
* FAISS create index
* save to S3

In [3]:
!conda install -y -c conda-forge pymupdf --quiet

Retrieving notices: ...working... done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /home/ec2-user/anaconda3/envs/python3

  added / updated specs:
    - pymupdf


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    curl-8.19.0                |       hcf29cc6_0         187 KB  conda-forge
    freeglut-3.2.2             |       ha6d2627_3         141 KB  conda-forge
    freetype-2.14.2            |       ha770c72_0         170 KB  conda-forge
    jbig2dec-0.18              |       h267a509_0          99 KB  conda-forge
    lerc-4.1.0                 |       hdb68285_0         255 KB  conda-forge
    libcurl-8.19.0             |       hcf29cc6_0         456 KB  conda-forge
    libfreetype-2.14.2         |       ha770c72_0           8 KB  conda-forge
    libfreetype6-2.14.2        |       h73754d4_0         377 

In [4]:
!conda install -y -c conda-forge faiss-cpu --quiet

Channels:
 - conda-forge
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /home/ec2-user/anaconda3/envs/python3

  added / updated specs:
    - faiss-cpu


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    faiss-1.9.0                |py312hf23773a_0_cpu         1.4 MB  conda-forge
    faiss-cpu-1.9.0            |       h718b53a_0          18 KB  conda-forge
    libfaiss-1.9.0             |   h72e5a87_0_cpu         1.6 MB  conda-forge
    numpy-1.26.4               |  py312heda63a1_0         7.1 MB  conda-forge
    ------------------------------------------------------------
                                           Total:        10.2 MB

The following NEW packages will be INSTALLED:

  faiss              conda-forge/linux-64::faiss-1.9.0-py312hf23773a_0_cpu 
  faiss-cpu          conda-forge/linux-64::faiss-cpu-1.9.0-h718b53a_0 
  libf

In [5]:
!pip install sentence-transformers boto3 --quiet

### Import Libs

In [6]:
import boto3
import fitz
print(fitz.__version__)
import json

import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

from io import BytesIO

1.24.1


### Configs

In [9]:
BUCKET_NAME = "human-design-knowledge-base"
PDF_PREFIX = "pdfs/"
INDEX_PREFIX = "index/"
CHUNK_SIZE = 800 # characters per chunk
CHUNK_OVERLAP = 100 # overlap between chunks
EMBEDDING_MODEL = "all-MiniLM-L6-v2" # 384 dimensions

s3 = boto3.client("s3")

### Step1: download PDFs from S3

In [10]:
def download_pdfs_from_s3(bucket, prefix):
    """Download all PDFs from S3 bucket to local /tmp directory"""
    local_dir = "/tmp/pdfs"
    os.makedirs(local_dir, exist_ok=True)
    
    response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    pdf_files = []
    
    for obj in response.get("Contents", []):
        key = obj["Key"]
        if key.endswith(".pdf"):
            filename = os.path.basename(key)
            local_path = os.path.join(local_dir, filename)
            print(f"Downloading: {key} ({obj['Size'] / 1024 / 1024:.1f} MB)")
            s3.download_file(bucket, key, local_path)
            pdf_files.append(local_path)
            print(f"  -> Saved to {local_path}")
    
    print(f"\nTotal PDFs downloaded: {len(pdf_files)}")
    return pdf_files

pdf_files = download_pdfs_from_s3(BUCKET_NAME, PDF_PREFIX)

Downloading: pdfs/Human Design_ Discover the Person You Were Born to Be - PDF Room.pdf (2.1 MB)
  -> Saved to /tmp/pdfs/Human Design_ Discover the Person You Were Born to Be - PDF Room.pdf
Downloading: pdfs/pdfcoffee.com_human-design-the-definitive-book-of-human-design-the-science-of-pdf-free.pdf (70.2 MB)
  -> Saved to /tmp/pdfs/pdfcoffee.com_human-design-the-definitive-book-of-human-design-the-science-of-pdf-free.pdf

Total PDFs downloaded: 2


In [11]:
pdf_files

['/tmp/pdfs/Human Design_ Discover the Person You Were Born to Be - PDF Room.pdf',
 '/tmp/pdfs/pdfcoffee.com_human-design-the-definitive-book-of-human-design-the-science-of-pdf-free.pdf']

### Step2: extract Text from PDFs

In [12]:
def extract_text_from_pdf(pdf_path):
    """Extract text from a PDF file using PyMuPDF"""
    doc = fitz.open(pdf_path)
    full_text = ""
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()
        if text.strip():
            full_text += text + "\n"
    
    doc.close()
    print(f"  Extracted {len(full_text):,} characters from {os.path.basename(pdf_path)}")
    return full_text

all_texts = {}
for pdf_path in pdf_files:
    print(f"\nProcessing: {os.path.basename(pdf_path)}")
    text = extract_text_from_pdf(pdf_path)
    all_texts[os.path.basename(pdf_path)] = text

total_chars = sum(len(t) for t in all_texts.values())
print(f"\n=== Total text extracted: {total_chars:,} characters from {len(all_texts)} PDFs ===")


Processing: Human Design_ Discover the Person You Were Born to Be - PDF Room.pdf
  Extracted 523,318 characters from Human Design_ Discover the Person You Were Born to Be - PDF Room.pdf

Processing: pdfcoffee.com_human-design-the-definitive-book-of-human-design-the-science-of-pdf-free.pdf
  Extracted 1,151,329 characters from pdfcoffee.com_human-design-the-definitive-book-of-human-design-the-science-of-pdf-free.pdf

=== Total text extracted: 1,674,647 characters from 2 PDFs ===


In [13]:
len(all_texts)

2

### Step3: chunk Text

In [14]:
def chunk_text(text, source_name, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """
    Split text into overlapping chunks with metadata.
    Uses character-based splitting with overlap to preserve context.
    """
    chunks = []
    start = 0
    chunk_id = 0
    
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        
        # try to break at sentence boundary (look for period, !, ?)
        if end < len(text):
            # search backwards from end for a sentence boundary
            for i in range(min(100, len(chunk) - 1), 0, -1):
                if chunk[-(i)] in ".!?\n":
                    chunk = chunk[:len(chunk) - i + 1]
                    end = start + len(chunk)
                    break
        
        # clean the chunk
        chunk_clean = chunk.strip()
        
        # only keep chunks with enough content (at least 50 characters)
        if len(chunk_clean) >= 50:
            chunks.append({
                "id": f"{source_name}_chunk_{chunk_id}",
                "text": chunk_clean,
                "source": source_name,
                "chunk_index": chunk_id
            })
            chunk_id += 1
        
        # move forward by (chunk_size - overlap)
        start = start + chunk_size - overlap
    
    return chunks

all_chunks = []
for source_name, text in all_texts.items():
    print(f"Chunking: {source_name}")
    chunks = chunk_text(text, source_name)
    all_chunks.extend(chunks)
    print(f"  -> {len(chunks)} chunks created")

print(f"\nTotal chunks: {len(all_chunks)}")

Chunking: Human Design_ Discover the Person You Were Born to Be - PDF Room.pdf
  -> 748 chunks created
Chunking: pdfcoffee.com_human-design-the-definitive-book-of-human-design-the-science-of-pdf-free.pdf
  -> 1645 chunks created

Total chunks: 2393


In [15]:
# preview first chunk
print(f"ID: {all_chunks[0]['id']}")
print(f"Source: {all_chunks[0]['source']}")
print(f"Length: {len(all_chunks[0]['text'])} chars")
print(f"Text preview: {all_chunks[0]['text'][:500]}...")

ID: Human Design_ Discover the Person You Were Born to Be - PDF Room.pdf_chunk_0
Source: Human Design_ Discover the Person You Were Born to Be - PDF Room.pdf
Length: 788 chars
Text preview: DOWNLOAD YOUR FREE SOFTWARE NOW!
 
This book is designed to be read alongside your Human Design life chart, which
reveals your personality blueprint.
 
To create your chart visit www.humandesignforusall.com. Follow the
instructions, download the Windows-based software, and take the first step on
your Human Design journey to find out who you really are.

A Revolutionary New System
Revealing the DNA of Your True Nature

HUMAN
DESIGN
 
Discover the Person
You Were Born to Be
CHETAN PARKYN
WITH STEV...


### Step4: generate Embeddings

In [19]:
# generate Embeddings
model = SentenceTransformer(EMBEDDING_MODEL)

# extract just the text for embedding
chunk_texts = [c["text"] for c in all_chunks]

print(f"Generating embeddings for {len(chunk_texts)} chunks...")
embeddings = model.encode(chunk_texts, show_progress_bar=True, batch_size=64)
print(f"Embeddings shape: {embeddings.shape}")
# expected: (num_chunks, 384)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings for 2393 chunks...


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Embeddings shape: (2393, 384)


### Step5: build FAISS Index

In [21]:
# build FAISS Index
dimension = embeddings.shape[1] # each chunk -> 384 dimention
print(f"Building FAISS index with dimension {dimension}")

# L2 (Euclidean) index — simple and effective for this scale
index = faiss.IndexFlatL2(dimension)

# normalize embeddings for better similarity search
faiss.normalize_L2(embeddings)

# add vectors to index
index.add(embeddings)

print(f"FAISS index built: {index.ntotal} vectors")

Building FAISS index with dimension 384
FAISS index built: 2393 vectors


In [22]:
# test Search (Sanity Check)
def search(query, model, index, chunks, top_k=5):
    """Search the FAISS index and return top-k results"""
    # encode query
    query_vec = model.encode([query])
    faiss.normalize_L2(query_vec)
    
    # search
    distances, indices = index.search(query_vec, top_k)
    
    results = []
    for i, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        results.append({
            "rank": i + 1,
            "score": 1 - dist,  # convert L2 distance to similarity
            "source": chunks[idx]["source"],
            "text": chunks[idx]["text"][:200]
        })
    return results

# test with a Human Design query
test_query = "What is a Generator type in Human Design?"
print(f"Test query: '{test_query}'\n")

results = search(test_query, model, index, all_chunks)
for r in results:
    print(f"  Rank {r['rank']} | Score: {r['score']:.3f} | Source: {r['source']}")
    print(f"  {r['text']}...")
    print()

Test query: 'What is a Generator type in Human Design?'

  Rank 1 | Score: 0.328 | Source: pdfcoffee.com_human-design-the-definitive-book-of-human-design-the-science-of-pdf-free.pdf
  Types. Learning to ask Generators good
'yes or no' questions makes all the difference.
Halfof the battle for Generators is accepting the fact that they cannot manifest their ideas for their
own life. ...

  Rank 2 | Score: 0.250 | Source: pdfcoffee.com_human-design-the-definitive-book-of-human-design-the-science-of-pdf-free.pdf
  70 percent of all people are Sacrally defined, the Generator's creative life-force
energy dominates the global frequency of the planet. Pure Generators comprise approximately 37
percent of humanity, a...

  Rank 3 | Score: 0.204 | Source: pdfcoffee.com_human-design-the-definitive-book-of-human-design-the-science-of-pdf-free.pdf
  ion. This is
why the Generator's mystical path and contribution to humanity is to wake up their Sacral, and to
live surrendered to its subjective truth.

### Step6: save to S3

In [23]:
# save to S3
LOCAL_INDEX_PATH = "/tmp/faiss_index.bin"
LOCAL_CHUNKS_PATH = "/tmp/chunks.json"

# save FAISS index
faiss.write_index(index, LOCAL_INDEX_PATH)
print(f"FAISS index saved: {os.path.getsize(LOCAL_INDEX_PATH) / 1024 / 1024:.2f} MB")

# save chunks metadata
with open(LOCAL_CHUNKS_PATH, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)
print(f"Chunks JSON saved: {os.path.getsize(LOCAL_CHUNKS_PATH) / 1024 / 1024:.2f} MB")

# upload to S3
s3.upload_file(LOCAL_INDEX_PATH, BUCKET_NAME, f"{INDEX_PREFIX}faiss_index.bin")
print(f"Uploaded: s3://{BUCKET_NAME}/{INDEX_PREFIX}faiss_index.bin")
s3.upload_file(LOCAL_CHUNKS_PATH, BUCKET_NAME, f"{INDEX_PREFIX}chunks.json")
print(f"Uploaded: s3://{BUCKET_NAME}/{INDEX_PREFIX}chunks.json")

print(f"Knowledge base built with {len(all_chunks)} chunks from {len(all_texts)} PDFs")
print(f"Files stored at s3://{BUCKET_NAME}/{INDEX_PREFIX}")

FAISS index saved: 3.51 MB
Chunks JSON saved: 2.29 MB
Uploaded: s3://human-design-knowledge-base/index/faiss_index.bin
Uploaded: s3://human-design-knowledge-base/index/chunks.json
Knowledge base built with 2393 chunks from 2 PDFs
Files stored at s3://human-design-knowledge-base/index/


In [25]:
# chunks.json
import json

obj = s3.get_object(Bucket=BUCKET_NAME, Key="index/chunks.json")
chunks_data = json.loads(obj["Body"].read().decode("utf-8"))

for i, chunk in enumerate(chunks_data[:3]):
    print(f"--- Chunk {i} ---")
    print(json.dumps(chunk, indent=2, ensure_ascii=False)[:500])
    print()

print(f"Total chunks: {len(chunks_data)}")

--- Chunk 0 ---
{
  "id": "Human Design_ Discover the Person You Were Born to Be - PDF Room.pdf_chunk_0",
  "text": "DOWNLOAD YOUR FREE SOFTWARE NOW!\n \nThis book is designed to be read alongside your Human Design life chart, which\nreveals your personality blueprint.\n \nTo create your chart visit www.humandesignforusall.com. Follow the\ninstructions, download the Windows-based software, and take the first step on\nyour Human Design journey to find out who you really are.\n\nA Revolutionary New System\nReveal

--- Chunk 1 ---
{
  "id": "Human Design_ Discover the Person You Were Born to Be - PDF Room.pdf_chunk_1",
  "text": "e or in part, stored in a retrieval system, or transmitted in any form or by any means —\nelectronic, mechanical, or other — without written permission from the publisher, except by a reviewer,\nwho may quote brief passages in a review.\nPage 7: “Your True Self © Margaret I. Jang 2005, www.onesourcelearn.com\nLibrary of Congress Cataloging-in-Publication Data Par